In [25]:
# CELL 1 — Config (masked-only, random 80/10/10)
import os, gc, glob, warnings, time, sys
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from skimage.measure import label as sk_label
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

DATA_DIR   = '/kaggle/input/competitions/uw-madison-gi-tract-image-segmentation'
TRAIN_DIR  = os.path.join(DATA_DIR, 'train')
TRAIN_CSV  = os.path.join(DATA_DIR, 'train.csv')

# Checkpoints directly under /kaggle/working (UI-friendly)
CKPT_BASELINE = '/kaggle/working/baseline_best.pth'
CKPT_S1       = '/kaggle/working/curriculum_stage1.pth'
CKPT_S2       = '/kaggle/working/curriculum_stage2.pth'
CKPT_S3       = '/kaggle/working/curriculum_stage3.pth'
MASK_CACHE    = '/kaggle/working/mask_cache'   # optional speedup

IMG_SIZE = 256
BATCH_SIZE = 16
BASELINE_EPOCHS = 35
STAGE_EPOCHS    = 25
LR_BASELINE   = 1e-4
LR_CURRICULUM = 1e-5
WEIGHT_DECAY  = 1e-2
EARLY_STOP    = 25
NUM_WORKERS   = min(4, os.cpu_count() or 0)
USE_MASK_CACHE = True

EASY_Q, HARD_Q = 0.60, 0.94
DIFFICULTY_THRESHOLDS = {}
EMPTY_PRED_THRESH = 0.01

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| Batch:', BATCH_SIZE)

Device: cuda | Batch: 16


In [2]:
# CELL 2 — Load data
train_df = pd.read_csv(TRAIN_CSV)
all_pngs = glob.glob(os.path.join(TRAIN_DIR, '**', '*.png'), recursive=True)

train_df['case_id']     = train_df['id'].apply(lambda x: int(x.split('_', 2)[0].replace('case', '')))
train_df['day_num']     = train_df['id'].apply(lambda x: int(x.split('_', 2)[1].replace('day', '')))
train_df['case_id_str'] = train_df['id'].apply(lambda x: x.split('_', 2)[0])
train_df['day_num_str'] = train_df['id'].apply(lambda x: x.split('_', 2)[1])
train_df['slice_id']    = train_df['id'].apply(lambda x: x.split('_', 2)[2])

train_df['_key'] = (
    TRAIN_DIR + '/' + train_df['case_id_str'] + '/' +
    train_df['case_id_str'] + '_' + train_df['day_num_str'] +
    '/scans/' + train_df['slice_id']
)
path_df = pd.DataFrame({'_key': [p.rsplit('_', 4)[0] for p in all_pngs], 'f_path': all_pngs})
train_df = train_df.merge(path_df, on='_key').drop(columns=['_key'])

train_df['slice_h'] = train_df['f_path'].apply(lambda x: int(x[:-4].rsplit('_', 4)[1]))
train_df['slice_w'] = train_df['f_path'].apply(lambda x: int(x[:-4].rsplit('_', 4)[2]))

for cls, col in [('large_bowel', 'lb_rle'), ('small_bowel', 'sb_rle'), ('stomach', 'st_rle')]:
    m = train_df[train_df['class'] == cls][['id', 'segmentation']].rename(columns={'segmentation': col})
    train_df = train_df.merge(m, on='id')
train_df = train_df.drop_duplicates('id').reset_index(drop=True)

for c in ['lb_rle', 'sb_rle', 'st_rle']:
    train_df[c.replace('_rle', '_flag')] = train_df[c].apply(lambda x: isinstance(x, str) and len(x) > 1)
train_df['has_mask'] = train_df[['lb_flag', 'sb_flag', 'st_flag']].any(axis=1)

train_df = train_df[[
    'id', 'f_path', 'slice_h', 'slice_w', 'lb_rle', 'sb_rle', 'st_rle',
    'has_mask', 'case_id', 'day_num'
]].reset_index(drop=True)

# MASKED ONLY for entire pipeline (conference scope: organ-present slices)
df_all = train_df[train_df['has_mask']].reset_index(drop=True)
print(f'Masked slices for study: {len(df_all)} / {len(train_df)} total')

Masked slices for study: 16590 / 38496 total


In [3]:
# CELL 2.5 — Precompute masks
def rle_decode(rle_str, h, w):
    mask = np.zeros(h * w, dtype=np.uint8)
    if not isinstance(rle_str, str) or not rle_str.strip():
        return mask.reshape(h, w)
    nums = list(map(int, rle_str.split()))
    for i in range(0, len(nums), 2):
        mask[nums[i] - 1: nums[i] - 1 + nums[i + 1]] = 1
    return mask.reshape(h, w, order='F')

def build_mask_row(row, size=256):
    h, w = int(row['slice_h']), int(row['slice_w'])
    ch = []
    for c in ['lb_rle', 'sb_rle', 'st_rle']:
        m = cv2.resize(rle_decode(row[c], h, w), (size, size), interpolation=cv2.INTER_NEAREST)
        ch.append(m)
    return np.stack(ch, 0).astype(np.float32)

if USE_MASK_CACHE:
    os.makedirs(MASK_CACHE, exist_ok=True)
    for _, row in tqdm(df_all.iterrows(), total=len(df_all), desc='Cache masks'):
        path = os.path.join(MASK_CACHE, row['id'] + '.npy')
        if not os.path.exists(path):
            np.save(path, build_mask_row(row, IMG_SIZE))
    print('Mask cache ready:', MASK_CACHE)
else:
    print('Mask cache skipped')

Cache masks: 100%|██████████| 16590/16590 [00:00<00:00, 23107.40it/s]

Mask cache ready: /kaggle/working/mask_cache


In [4]:
# CELL 3 — Random 80/10/10 on MASKED slices only
idx = np.arange(len(df_all))

tr_idx, temp_idx = train_test_split(idx, test_size=0.20, random_state=42)
va_idx, te_idx = train_test_split(temp_idx, test_size=0.50, random_state=42)

df_train = df_all.iloc[tr_idx].sample(frac=1, random_state=42).reset_index(drop=True)
df_val   = df_all.iloc[va_idx].reset_index(drop=True)
df_test  = df_all.iloc[te_idx].reset_index(drop=True)   # use only in Cell 14

print(f'Train {len(df_train)} | Val {len(df_val)} | Test {len(df_test)}')
print(f'Batches/epoch @ BS={BATCH_SIZE}: {len(df_train)//BATCH_SIZE + 1}')
print(f'Case overlap train∩test: {len(set(df_train.case_id)&set(df_test.case_id))} (state in limitations)')

Train 13272 | Val 1659 | Test 1659
Batches/epoch @ BS=16: 830
Case overlap train∩test: 85 (state in limitations)


In [5]:
# CELL 4 — RLE, composite, metrics
def load_image(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(path)
    return img.astype(np.float32) / 65535.0

def build_mask(row):
    if USE_MASK_CACHE:
        return np.load(os.path.join(MASK_CACHE, row['id'] + '.npy'))
    return build_mask_row(row, IMG_SIZE)

def normalize_img(img):
    mn, mx = img.min(), img.max()
    if mx > mn:
        img = (img - mn) / (mx - mn)
    return (img - 0.449) / 0.226

def compute_composite_score(img_gray, mask_union, baseline_dice):
    area = float(mask_union.sum())
    S1 = 1.0 - (area / mask_union.size) if area > 0 else 1.0
    hist = np.histogram(np.clip(img_gray, 0, 1).ravel(), bins=256, range=(0, 1))[0].astype(float)
    p = hist[hist > 0]
    if len(p):
        p /= p.sum()
        S2 = min(-np.sum(p * np.log2(p)) / 8.0, 1.0)
    else:
        S2 = 0.0
    if area > 0:
        n_comp = len(np.unique(sk_label(mask_union > 0))) - 1
        S3 = min(n_comp / 20.0, 1.0)
    else:
        S3 = 0.0
    S4 = 1.0 - float(np.clip(baseline_dice, 0.0, 1.0))
    return (S1 + S2 + S3 + S4) / 4.0

def label_difficulty(score, easy_lim, hard_lim):
    if score < easy_lim:
        return 'Easy'
    if score >= hard_lim:
        return 'Hard'
    return 'Medium'

def dice_metric_A(logits, targets, smooth=1.0, empty_thr=EMPTY_PRED_THRESH):
    """Per-channel hard Dice; empty channel GT → 1 if pred empty."""
    pred = (torch.sigmoid(logits) > 0.5).float()
    B, C = pred.shape[0], pred.shape[1]
    p = pred.view(B, C, -1)
    t = targets.float().view(B, C, -1)
    inter = (p * t).sum(-1)
    denom = p.sum(-1) + t.sum(-1)
    dice = (2 * inter + smooth) / (denom + smooth)
    empty_gt = t.sum(-1) == 0
    dice = torch.where(empty_gt, (p.sum(-1) < empty_thr).float(), dice)
    return dice.mean().item()

def dice_metric_B(logits, targets, threshold=0.5, smooth=1e-6):
    """PS metric: per-slice mean over present organ channels."""
    pred = (torch.sigmoid(logits) > threshold).float()
    p = pred.view(pred.size(0), pred.size(1), -1)
    t = targets.float().view(pred.size(0), targets.size(1), -1)
    inter = (p * t).sum(-1)
    denom = p.sum(-1) + t.sum(-1)
    d = (2 * inter + smooth) / (denom + smooth)
    ok = t.sum(-1) > 0
    per_slice = []
    for b in range(pred.size(0)):
        if ok[b].any():
            per_slice.append(d[b][ok[b]].mean().item())
    return float(np.mean(per_slice)) if per_slice else None

In [6]:
# CELL 5 — Dataset
class GIDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = normalize_img(cv2.resize(load_image(row['f_path']), (IMG_SIZE, IMG_SIZE)))
        mask = build_mask(row)
        if self.augment:
            if np.random.rand() > 0.5:
                img, mask = img[:, ::-1].copy(), mask[:, :, ::-1].copy()
            if np.random.rand() > 0.5:
                img, mask = img[::-1, :].copy(), mask[:, ::-1, :].copy()
        return torch.from_numpy(img[None].astype(np.float32)), torch.from_numpy(mask)

class GIDatasetEval(GIDataset):
    def __getitem__(self, idx):
        img, mask = super().__getitem__(idx)
        return img, mask, self.df.iloc[idx]['difficulty']

In [26]:
# CELL 6 — AGTSNet
class ConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class AGTSNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=3):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.enc1 = ConvBnRelu(in_ch, 64)
        self.enc2 = ConvBnRelu(64, 128)
        self.enc3 = ConvBnRelu(128, 256)
        self.enc4 = ConvBnRelu(256, 512)
        self.bottleneck = ConvBnRelu(512, 1024, dropout=0.15)
        self.dec4 = ConvBnRelu(1024 + 512, 512, dropout=0.1)
        self.dec3 = ConvBnRelu(512 + 256, 256)
        self.dec2 = ConvBnRelu(256 + 128, 128)
        self.dec1 = ConvBnRelu(128 + 64, 64)
        self.head = nn.Conv2d(64, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d = F.interpolate(b, size=e4.shape[2:], mode='bilinear', align_corners=False)
        d = self.dec4(torch.cat([d, e4], dim=1))
        d = F.interpolate(d, size=e3.shape[2:], mode='bilinear', align_corners=False)
        d = self.dec3(torch.cat([d, e3], dim=1))
        d = F.interpolate(d, size=e2.shape[2:], mode='bilinear', align_corners=False)
        d = self.dec2(torch.cat([d, e2], dim=1))
        d = F.interpolate(d, size=e1.shape[2:], mode='bilinear', align_corners=False)
        d = self.dec1(torch.cat([d, e1], dim=1))
        return self.head(d)

def unwrap(model):
    return model.module if isinstance(model, nn.DataParallel) else model

In [8]:
# CELL 7 — Focal + Dice
class DiceLoss(nn.Module):
    def forward(self, logits, targets):
        p = torch.sigmoid(logits).view(logits.size(0), logits.size(1), -1)
        t = targets.float().view(targets.size(0), targets.size(1), -1)
        inter = (p * t).sum(2)
        denom = p.sum(2) + t.sum(2)
        dice = (2 * inter + 1e-6) / (denom + 1e-6)
        ok = t.sum(2) > 0
        if not ok.any():
            return logits.sum() * 0
        return 1 - dice[ok].mean()

class FocalLoss(nn.Module):
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets.float(), reduction='none')
        pt = torch.exp(-bce)
        at = targets * 0.75 + (1 - targets) * 0.25
        return (at * (1 - pt) ** 2 * bce).mean()

class CombinedLoss(nn.Module):
    def forward(self, logits, targets):
        return 0.5 * DiceLoss()(logits, targets) + 0.5 * FocalLoss()(logits, targets)

In [9]:
# CELL 8 — Train (best checkpoint on val Metric A; log B)
def run_epoch(model, loader, optimizer, loss_fn, device, train=True, scaler=None):
    model.train() if train else model.eval()
    tl, n_loss = 0.0, 0
    sum_a, sum_b, n_a, n_b = 0.0, 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()

    with ctx:
        desc = 'Train' if train else 'Val  '
        for imgs, masks in tqdm(loader, desc=desc, leave=True, file=sys.stdout, mininterval=0.3):
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast():
                    logits = model(imgs)
                    loss = loss_fn(logits, masks)
                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                with torch.cuda.amp.autocast():
                    logits = model(imgs)
                    loss = loss_fn(logits, masks)

            tl += loss.item()
            n_loss += 1
            sum_a += dice_metric_A(logits.detach(), masks.detach())
            n_a += 1
            db = dice_metric_B(logits.detach(), masks.detach())
            if db is not None:
                sum_b += db
                n_b += 1

            postfix = {'loss': f'{tl/n_loss:.4f}', 'dA': f'{sum_a/n_a:.4f}'}
            if n_b > 0:
                postfix['dB'] = f'{sum_b/n_b:.4f}'
            # tqdm postfix omitted here for brevity in loop — add if you want live bar

    return tl / n_loss, sum_a / n_a, (sum_b / n_b if n_b else 0.0)

def train_model(model, df_tr, df_va, n_epochs, lr, ckpt_path, patience=EARLY_STOP):
    tr_loader = DataLoader(
        GIDataset(df_tr, True), batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0)
    )
    va_loader = DataLoader(
        GIDataset(df_va, False), batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0)
    )

    loss_fn = CombinedLoss()
    scaler = torch.cuda.amp.GradScaler()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=lr * 0.01)

    best_a, wait, history = -1.0, 0, []

    for epoch in range(1, n_epochs + 1):
        print(f'\n── Epoch {epoch}/{n_epochs} ──', flush=True)
        tr_loss, tr_a, tr_b = run_epoch(model, tr_loader, optimizer, loss_fn, DEVICE, True, scaler)
        va_loss, va_a, va_b = run_epoch(model, va_loader, None, loss_fn, DEVICE, False, None)
        scheduler.step()

        history.append({
            'epoch': epoch, 'tr_dice_A': tr_a, 'va_dice_A': va_a,
            'tr_dice_B': tr_b, 'va_dice_B': va_b
        })

        gap = tr_a - va_a
        if va_a > best_a:
            best_a = va_a
            torch.save(unwrap(model).state_dict(), ckpt_path)
            wait = 0
            mk = '  <- BEST'
        else:
            wait += 1
            mk = ''

        print(
            f'Ep {epoch:3d} | A {tr_a:.4f}/{va_a:.4f} | B {tr_b:.4f}/{va_b:.4f} | '
            f'gap_A {gap:.4f}{mk}', flush=True
        )
        print(f'  saved: {ckpt_path}', flush=True)

        if wait >= patience:
            print(f'Early stop. Best val A: {best_a:.4f}', flush=True)
            break

    gc.collect()
    torch.cuda.empty_cache()
    return history, best_a

In [ ]:
# CELL 9 — Baseline → /kaggle/working/baseline_best.pth
model = AGTSNet().to(DEVICE)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

hist, best_a = train_model(model, df_train, df_val, BASELINE_EPOCHS, LR_BASELINE, CKPT_BASELINE)
print(f'\nDone. Best val Dice A: {best_a:.4f}')
print(f'Download: {CKPT_BASELINE}')

In [11]:
# CELL 10 — Composite score (train masked)
CKPT_BASELINE = '/kaggle/input/baseline_best.pth'

model = AGTSNet().to(DEVICE)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

unwrap(model).load_state_dict(torch.load(CKPT_BASELINE, map_location=DEVICE))
model.eval()
print('Baseline loaded.')

loader = DataLoader(GIDataset(df_train, False), BATCH_SIZE, False, num_workers=NUM_WORKERS)
s4_list = []
with torch.no_grad():
    for imgs, masks in tqdm(loader, desc='S4'):
        logits = model(imgs.to(DEVICE))
        pred = (torch.sigmoid(logits) > 0.5).cpu().numpy()
        for b in range(pred.shape[0]):
            ch = []
            for c in range(3):
                g, p = masks[b, c].numpy(), pred[b, c]
                if g.sum() == 0:
                    continue
                ch.append((2 * (p * g).sum() + 1e-6) / (p.sum() + g.sum() + 1e-6))
            s4_list.append(float(np.mean(ch)) if ch else 0.0)

scores = []
for i, row in tqdm(df_train.iterrows(), total=len(df_train), desc='Composite'):
    img = cv2.resize(load_image(row['f_path']), (IMG_SIZE, IMG_SIZE))
    mn, mx = img.min(), img.max()
    if mx > mn:
        img = (img - mn) / (mx - mn)
    union = (build_mask(row).sum(axis=0) > 0).astype(np.uint8)
    scores.append(compute_composite_score(img, union, s4_list[i]))

df_scored = df_train.copy()
df_scored['difficulty_score'] = scores

easy_lim = df_scored['difficulty_score'].quantile(EASY_Q)
hard_lim = df_scored['difficulty_score'].quantile(HARD_Q)
DIFFICULTY_THRESHOLDS['easy'] = easy_lim
DIFFICULTY_THRESHOLDS['hard'] = hard_lim

df_scored['difficulty'] = df_scored['difficulty_score'].apply(
    lambda s: label_difficulty(s, easy_lim, hard_lim)
)
print(df_scored['difficulty'].value_counts())

df_easy   = df_scored[df_scored['difficulty'] == 'Easy'].reset_index(drop=True)
df_medium = df_scored[df_scored['difficulty'] == 'Medium'].reset_index(drop=True)
df_hard   = df_scored[df_scored['difficulty'] == 'Hard'].reset_index(drop=True)

Baseline loaded.


Composite: 100%|██████████| 13272/13272 [01:12<00:00, 183.24it/s]

difficulty
Easy      7963
Medium    4512
Hard       797
Name: count, dtype: int64


In [12]:
# CELL 11 — PS stages (masked tiers)
df_stage1 = df_easy.copy()
df_stage2 = pd.concat([df_easy, df_medium], ignore_index=True).reset_index(drop=True)
df_stage3 = pd.concat([df_easy, df_medium, df_hard], ignore_index=True).reset_index(drop=True)
print(len(df_stage1), len(df_stage2), len(df_stage3))

7963 12475 13272


In [ ]:
# Free the scoring model from Cell 10 before curriculum training
del model
gc.collect()
torch.cuda.empty_cache()

# starting curriculum
model_curr = AGTSNet().to(DEVICE)
model_curr.load_state_dict(torch.load(CKPT_BASELINE, map_location=DEVICE))

In [ ]:
# CELL 12 — Curriculum
gc.collect()
torch.cuda.empty_cache()

CKPT_BASELINE = '/kaggle/input/datasets/mythilinandanavanam/gitract-baseline/baseline_best.pth'

model_curr = AGTSNet().to(DEVICE)
model_curr.load_state_dict(torch.load(CKPT_BASELINE, map_location=DEVICE))

print('STAGE 1'); train_model(model_curr, df_stage1, df_val, STAGE_EPOCHS, LR_CURRICULUM, CKPT_S1, 8)
model_curr.load_state_dict(torch.load(CKPT_S1, map_location=DEVICE))

print('Saved:', CKPT_S1)

In [ ]:
# STAGE 2
model_curr = AGTSNet().to(DEVICE)
model_curr.load_state_dict(torch.load(CKPT_S1, map_location=DEVICE))
print('STAGE 2')
train_model(model_curr, df_stage2, df_val, STAGE_EPOCHS, LR_CURRICULUM, CKPT_S2, 8)

In [ ]:
import os
import gc
import torch

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# STAGE 3
model_curr = AGTSNet().to(DEVICE)
model_curr.load_state_dict(torch.load(CKPT_S2, map_location=DEVICE))
print('STAGE 3')
train_model(model_curr, df_stage3, df_val, STAGE_EPOCHS, LR_CURRICULUM, CKPT_S3, 25)
print('Saved:', CKPT_S3)

In [ ]:
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
torch.cuda.empty_cache()

model_curr = AGTSNet().to(DEVICE)
model_curr.load_state_dict(torch.load(CKPT_S3, map_location=DEVICE))
train_model(model_curr, df_stage3, df_val, 23, LR_CURRICULUM, CKPT_S3, 25)
print('Saved:', CKPT_S3)

In [ ]:
import gc
import torch

# 1. Clear GPU memory completely
gc.collect()
torch.cuda.empty_cache()

# 2. Re-initialize the model structure
model_curr = AGTSNet().to(DEVICE)

# 3. Load the checkpoint saved from your current run
model_curr.load_state_dict(torch.load(CKPT_S3, map_location=DEVICE))
print('STAGE 3: Resuming from the current checkpoint')

# 4. Run exactly 16 more epochs to reach the total target of 25
train_model(model_curr, df_stage3, df_val, 16, LR_CURRICULUM, CKPT_S3, 25)

print('Saved final curriculum weights:', CKPT_S3)

In [23]:
# CELL 13 — Score test set with baseline difficulty labels
def score_test(df_te, ckpt):
    m = AGTSNet().to(DEVICE)
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    m.eval()
    ld = DataLoader(GIDataset(df_te, False), BATCH_SIZE, False,
                    num_workers=NUM_WORKERS)
    s4_list = []
    with torch.no_grad():
        for imgs, masks in tqdm(ld, desc='S4 test'):
            logits = m(imgs.to(DEVICE))
            pred   = (torch.sigmoid(logits) > 0.5).cpu().numpy()
            
            # FIXED SYNTAX: Added .shape[0] to prevent loop corruption
            for b in range(pred.shape[0]):
                ch = []
                for c in range(3):
                    g, p = masks[b, c].numpy(), pred[b, c]
                    if g.sum() == 0:
                        continue
                    ch.append((2*(p*g).sum()+1e-6)/(p.sum()+g.sum()+1e-6))
                s4_list.append(float(np.mean(ch)) if ch else 0.0)

    scores = []
    for (_, row), d in zip(df_te.iterrows(), s4_list):
        img   = cv2.resize(load_image(row['f_path']), (IMG_SIZE, IMG_SIZE))
        union = (build_mask(row).sum(axis=0) > 0).astype(np.uint8)
        scores.append(compute_composite_score(img, union, d))

    scores_arr = np.array(scores)
    print(f'Score distribution on test set:')
    print(f'  min={scores_arr.min():.3f}  max={scores_arr.max():.3f}  mean={scores_arr.mean():.3f}')
    
    # RE-ENABLE QUANTILE MAPPING: Guarantees exactly 60% Easy and 6% Hard slices
    easy_lim_test = float(np.percentile(scores_arr, 60))
    hard_lim_test = float(np.percentile(scores_arr, 94))
    print(f'  Test thresholds (60th/94th percentiles) — easy_lim={easy_lim_test:.3f}  hard_lim={hard_lim_test:.3f}')

    out = df_te.copy()
    out['difficulty_score'] = scores
    out['difficulty'] = out['difficulty_score'].apply(
        lambda s: label_difficulty(s, easy_lim_test, hard_lim_test)
    )
    del m; gc.collect(); torch.cuda.empty_cache()
    return out

df_test_scored = score_test(df_test, CKPT_BASELINE)
vc = df_test_scored['difficulty'].value_counts()
print('\nTest set difficulty split:')
for tier in ['Easy', 'Medium', 'Hard']:
    count = vc.get(tier, 0)
    print(f'  {tier:8s}: {count:>5d}  ({100*count/len(df_test_scored):.1f}%)')

S4 test: 100%|██████████| 104/104 [00:32<00:00,  3.21it/s]


Score distribution on test set:
  min=0.286  max=0.684  mean=0.454
  Test thresholds (60th/94th percentiles) — easy_lim=0.472  hard_lim=0.559

Test set difficulty split:
  Easy    :   995  (60.0%)
  Medium  :   564  (34.0%)
  Hard    :   100  (6.0%)


In [24]:
# CELL 14 — Final evaluation: Baseline vs Curriculum (FULL TEST set)
def eval_metric_B_full(model_path, df_sc):
    """
    Metric B: per-slice Dice over PRESENT organ channels only.
    Evaluates on EVERY SINGLE available slice per tier to eliminate sampling bias.
    """
    m = AGTSNet().to(DEVICE)
    m.load_state_dict(torch.load(model_path, map_location=DEVICE))
    m.eval()

    vc = df_sc['difficulty'].value_counts()
    print(f'  Evaluating full dataset across all tiers '
          f'(Total Slices: Easy={vc.get("Easy",0)}, Medium={vc.get("Medium",0)}, Hard={vc.get("Hard",0)})')

    ld = DataLoader(GIDatasetEval(df_sc.reset_index(drop=True), False), BATCH_SIZE, False,
                    num_workers=NUM_WORKERS)
    cohort = {'Easy': [], 'Medium': [], 'Hard': []}

    with torch.no_grad():
        for imgs, masks, tiers in tqdm(ld, desc=f'MetricB {os.path.basename(model_path)[:20]}'):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            logits = m(imgs)
            pred   = (torch.sigmoid(logits) > 0.5).float()
            p_flat = pred.view(pred.size(0), 3, -1)
            t_flat = masks.float().view(masks.size(0), 3, -1)
            inter  = (p_flat * t_flat).sum(-1)
            denom  = p_flat.sum(-1) + t_flat.sum(-1)
            d      = (2 * inter + 1e-6) / (denom + 1e-6)
            ok     = t_flat.sum(-1) > 0
            for b, tier in enumerate(tiers):
                if ok[b].any():
                    cohort[tier].append(d[b][ok[b]].mean().item())

    res = {k: float(np.mean(v)) if v else 0.0 for k, v in cohort.items()}
    res['Balanced'] = float(np.mean([res['Easy'], res['Medium'], res['Hard']]))
    del m; gc.collect(); torch.cuda.empty_cache()
    return res


# Run Full Evaluation
print('=' * 65)
print('  FINAL RESULTS — FULL UN-BIASED TEST EVALUATION')
print('  Metric B: per-slice Dice on present organ channels only')
print('=' * 65)

base_B = eval_metric_B_full(CKPT_BASELINE, df_test_scored)
curr_B = eval_metric_B_full(CKPT_S3,       df_test_scored)

print(f'\n{"Tier":12s} | {"Baseline":>10s} | {"Curriculum":>10s} | {"Δ pts":>8s} | {"Δ rel%":>8s}')
for tier in ['Easy', 'Medium', 'Hard', 'Balanced']:
    b, c = base_B[tier], curr_B[tier]
    d_pts = (c - b) * 100
    d_rel = (c - b) / max(b, 1e-6) * 100
    if tier == 'Balanced':
        print('-' * 58)
    print(f'{tier:12s} | {b:10.4f} | {c:10.4f} | {d_pts:+8.2f} | {d_rel:+8.2f}%')

  FINAL RESULTS — FULL UN-BIASED TEST EVALUATION
  Metric B: per-slice Dice on present organ channels only
  Evaluating full dataset across all tiers (Total Slices: Easy=995, Medium=564, Hard=100)


MetricB baseline_best.pth: 100%|██████████| 104/104 [00:31<00:00,  3.29it/s]


  Evaluating full dataset across all tiers (Total Slices: Easy=995, Medium=564, Hard=100)


MetricB curriculum_stage3.pt: 100%|██████████| 104/104 [00:31<00:00,  3.33it/s]



Tier         |   Baseline | Curriculum |    Δ pts |   Δ rel%
----------------------------------------------------------
Easy         |     0.6621 |     0.7251 |    +6.30 |    +9.51%
Medium       |     0.5430 |     0.6105 |    +6.76 |   +12.44%
Hard         |     0.4401 |     0.5102 |    +7.01 |   +15.92%
----------------------------------------------------------
Balanced     |     0.5484 |     0.6153 |    +6.69 |   +12.19%
